# LLM 8일차 (3) — 지난 시간 검색 분기 에이전트의 수업 실행 기록 읽기 (`LangGraph_실습3_1`)

`LangGraph_실습3_1` 은 Day 052 의 `LangGraph_실습3`(라우팅 함수에서 멈춤)을 수업에서 **완성 · 실제 실행한 판**이다. 코드는 Day 052 정리 노트 [`langgraph_search_router_agent.ipynb`](../../2026-11-20/lecture/langgraph_search_router_agent.ipynb) 와 같은 구조라 다시 쓰지 않고, **진짜 gpt-5-nano · Tavily 로 돌린 출력**을 옮겨 확인했다.

> **출력 읽는 법** — 아래 두 셀은 모두 `# 📼` (수업 시간 출력을 그대로 옮김). 코드는 실행하지 않았다(키 필요).

## 1. 필기의 완성 부분 (Day 052 정리 노트와 비교)

| 부분 | `LangGraph_실습3_1` | 비고 |
|---|---|---|
| 라우팅 함수 | `route_by_search_need(state) -> str` + `print('[Router] …')` | 매핑 dict 를 함께 줘서 그림도 바르게 그려진다 |
| 그래프 | `set_entry_point('classifier_node')` · `add_conditional_edges` · `add_edge('search_node', 'answer_node')` · `add_edge('answer_node', END)` | 교재 p.101 과 같다 (교재의 `work.` 오타 없음) |
| 실행 | `agent.invoke(AgentState(messages=[HumanMessage(...)]))` → `final_state['messages'][-1].content` | BaseModel 객체를 넣어도 결과는 dict |
| 남은 문제 | `load_dotenv()` 호출 없음 · `== "SEARCH"` 비교 · 검색 실패 시 `UnboundLocalError` · 변수 이름 `inital_state1` 오타 | Day 052 정리 노트에서 고침 (수업 PC 는 환경 변수에 키가 있어 `load_dotenv` 없이도 돌았던 것으로 보인다) |

## 2. 테스트 1 — 일반 지식 질문

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

test_query1 = "파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘."
final_state1 = agent.invoke(AgentState(messages=[HumanMessage(content=test_query1)]))
print(f"[최종 AI 답변]: \n{final_state1['messages'][-1].content}")

테스트 1: 일반 지식 질문 (웹 검색 불필요 -> 즉시 답변 직행 경로 검증)
[Node 1: 질문 의도 분류] 실행 중...
--------------------------------------------------------------------------------
state: messages=[HumanMessage(content='파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘.', additional_kwargs={}, response_metadata={}, id='125c7c6e-0d65-4e10-a83b-b053ce867e94')] needs_search=False search_results=''
--------------------------------------------------------------------------------
[사용자 질문 원본]: 파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘.
--------------------------------------------------------------------------------
llm의 답변: 
content='NO_SEARCH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 174, 'total_tokens': 249, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, '

## 3. 테스트 2 — 최신 정보 질문

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

test_query2 = "Nvidia 2026년 2분기 영업 실적 요약해 줘."
final_state2 = agent.invoke(AgentState(messages=[HumanMessage(content=test_query2)]))
print(f"[최종 AI 답변]:\n{final_state2['messages'][-1].content}")

테스트 2: 실시간 최신 정보 질문 (웹 검색 필요 -> 검색 -> 답변 경로 검증)
[Node 1: 질문 의도 분류] 실행 중...
--------------------------------------------------------------------------------
state: messages=[HumanMessage(content='Nvidia 2026년 2분기 영업 실적 요약해 줘.', additional_kwargs={}, response_metadata={}, id='aac3ce59-fa91-44a6-b56b-de0c3b913645')] needs_search=False search_results=''
--------------------------------------------------------------------------------
[사용자 질문 원본]: Nvidia 2026년 2분기 영업 실적 요약해 줘.
--------------------------------------------------------------------------------
llm의 답변: 
content='SEARCH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 266, 'prompt_tokens': 167, 'total_tokens': 433, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None,

## 4. 출력에서 확인한 것

| 확인 | 결과 |
|---|---|
| 실행 경로 | 테스트 1 `classifier → answer` (Router : 즉시 answer_node) · 테스트 2 `classifier → search → answer` (검색 3건) — **조건부 분기가 의도대로** 동작 |
| 분류 출력 | `content='NO_SEARCH'` · `content='SEARCH'` — 이번에는 따옴표 없이 정확히 나와 `==` 비교가 통했다. 그래도 Day 052 처럼 앞뒤 기호를 떼는 편이 안전하다 |
| 분류 비용 | 단어 하나를 답하는 데 `reasoning_tokens` **64개 · 256개** — gpt-5-nano 는 추론 모델이라 답하기 전에 '생각' 토큰을 쓴다. 분류처럼 쉬운 일은 `ChatOpenAI(model='gpt-5-nano', reasoning_effort='minimal')` 로 줄일 수 있다(매개변수는 langchain-openai 1.7.4 에 있음 · 효과는 직접 확인 필요) |
| 노드 출력 | `state: … needs_search=False search_results=''` — 노드는 `AgentState` 객체를 받는다 (Day 051) |

### 테스트 2 답의 숫자 — NVIDIA 보도자료와 대조 (2026-08-26 발표, Q2 FY2027)

| 항목 | 답변 | 보도자료 | |
|---|---|---|---|
| 분기 | FY27 2분기 · 2026-07-26 종료 | 2026-07-26 종료 | ✅ |
| 매출 | 962.2억 달러 · QoQ +18% · YoY +106% | $96,221M · +18% · +106% | ✅ |
| 데이터센터 | 890억 달러 · +18% · +117% | $89.0B · +18% · +117% | ✅ |
| 매출총이익률 | 75.0% (GAAP · 비GAAP 같음) | 75.0% · 75.0% | ✅ |
| 매출원가 · 총이익 | 240.79억 · 721.42억 | $24,079M · (96,221 − 24,079 = 72,142) | ✅ |
| 영업이익 | 637.34억 · +19% · +124% | GAAP $63,734M · +19% · +124% | ✅ |
| 순이익 | 596.88억 · +2% · +126% | GAAP $59,688M · +2% · +126% | ✅ |
| EPS | GAAP 2.46 · 비GAAP 2.22 | $2.46 · $2.22 | ✅ |
| 주주환원 | 약 260억 · 남은 자사주 매입 한도 약 990억 | 약 $26.0B · 약 $99.0B | ✅ |
| 배당 | 주당 0.25달러 · 2026-10-01 지급 | $0.25 · 2026-10-01 | ✅ |

- 📝 숫자는 모두 맞았다. 다만 읽을 때 주의할 점 두 가지
  1. **'2026년 2분기' 가 무엇인가** — NVIDIA 회계연도는 1월 말에 끝나서 'FY2027 2분기' = **2026년 5~7월**이다. 달력 2분기(4~6월)와 다르고, '회계 2026년 2분기'(FY2026 Q2, 2025년 7월 종료, 매출 $46.7B)로 읽을 수도 있다. 답이 'FY27 2분기, 7월 26일 종료' 라고 밝힌 것은 좋았다 — 질문을 만들 때 기준을 적어 주는 것이 안전하다
  2. **GAAP 순이익이 영업이익 증가(+19%)와 달리 +2%** 인 것, **GAAP EPS(2.46) > 비GAAP EPS(2.22)** 인 것은 오류가 아니다. 비GAAP 은 투자 지분 평가이익 같은 영업 외 손익을 빼고 계산해서, 1분기 GAAP 순이익이 이미 컸던 탓에 QoQ 가 작게 나온다(비GAAP 순이익은 QoQ +18%). 또 FY27 부터 NVIDIA 비GAAP 은 주식보상비용을 **빼지 않도록** 바뀌었다(보도자료 주석)
- 마지막 줄 '(IF FY27 Q2)' 는 뜻이 없는 표현 — LLM 이 만든 군더더기다

출처 : [NVIDIA Announces Financial Results for Second Quarter Fiscal 2027](https://investor.nvidia.com/news/press-release-details/2026/NVIDIA-Announces-Financial-Results-for-Second-Quarter-Fiscal-2027/)